## Read nd2 files, store them in a folder, and name each channel and position appropriately (XY... or C...)

In [1]:
import os
from nd2reader import ND2Reader
from PIL import Image
import concurrent.futures
from tqdm import tqdm  # For progress bar (optional)
import threading

import gc

def export_channel_field(nd2_filepath, output_directory, channel_idx, field_idx, width, height, progress_bar=None):
    with ND2Reader(nd2_filepath) as images:
        images.default_coords['v'] = field_idx
        images.default_coords['c'] = channel_idx
        try:
            frame = images.get_frame_2D(c=channel_idx, v=field_idx)
            if frame.shape == (height, width):
                base_filename = os.path.basename(nd2_filepath)
                filename_without_ext = os.path.splitext(base_filename)[0]
                output_filename = f"{filename_without_ext}_XY{field_idx}_C{channel_idx + 1}.tiff"
                output_filepath = os.path.join(output_directory, output_filename)
                Image.fromarray(frame).save(output_filepath)
                if progress_bar is not None:
                    progress_bar.update(1)
            else:
                print(f"Skipping invalid frame for field {field_idx}, channel {channel_idx}")
        except KeyError as e:
            print(f"KeyError for field {field_idx}, channel {channel_idx}: {e}")

def export_channels_to_tiff_parallel(nd2_filepath, output_directory, max_workers=2):  # Use fewer workers
    os.makedirs(output_directory, exist_ok=True)
    with ND2Reader(nd2_filepath) as images:
        metadata = images.metadata
        num_channels = images.sizes.get('c', 1)
        num_fields = images.sizes.get('v', 1)
        width = images.sizes.get('x', None)
        height = images.sizes.get('y', None)
        
        total_tasks = num_channels * num_fields
        with tqdm(total=total_tasks, desc=f"Processing {os.path.basename(nd2_filepath)}") as progress_bar:
            with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
                futures = []
                for field_idx in range(num_fields):
                    for channel_idx in range(num_channels):
                        futures.append(
                            executor.submit(
                                export_channel_field, nd2_filepath, output_directory,
                                channel_idx, field_idx, width, height, progress_bar
                            )
                        )
                for future in concurrent.futures.as_completed(futures):
                    future.result()
    # Explicitly call garbage collection after processing each file
    gc.collect()

def process_nd2_files_parallel(input_directory, base_output_directory, max_workers=2):
    os.makedirs(base_output_directory, exist_ok=True)
    nd2_files = [f for f in os.listdir(input_directory) if f.endswith('.nd2')]
    
    # Process files in batches (e.g., one file at a time) to limit resource usage
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = []
        for nd2_file in nd2_files:
            nd2_filepath = os.path.join(input_directory, nd2_file)
            output_directory = os.path.join(base_output_directory, os.path.splitext(nd2_file)[0])
            futures.append(
                executor.submit(export_channels_to_tiff_parallel, nd2_filepath, output_directory, max_workers)
            )
        for future in concurrent.futures.as_completed(futures):
            future.result()
    gc.collect()

In [ ]:
#aangepaste export march 25

In [4]:
#zelf
def export_channel_field(nd2_filepath, output_directory, channel_idx, field_idx, width, height, progress_bar=None):
    # 1. Define the filename first
    base_filename = os.path.basename(nd2_filepath)
    filename_without_ext = os.path.splitext(base_filename)[0]
    output_filename = f"{filename_without_ext}_XY{field_idx}_C{channel_idx + 1}.tiff"
    output_filepath = os.path.join(output_directory, output_filename)

    # 2. CHECK IF EXISTS: Skip the entire function if the file is already there
    if os.path.exists(output_filepath):
        if progress_bar is not None:
            progress_bar.update(1)
        return  # Stop here and move to the next image

    # 3. Only open the file if the image is missing
    with ND2Reader(nd2_filepath) as images:
        images.default_coords['v'] = field_idx
        images.default_coords['c'] = channel_idx
        try:
            frame = images.get_frame_2D(c=channel_idx, v=field_idx)
            if frame.shape == (height, width):
                Image.fromarray(frame).save(output_filepath)
                if progress_bar is not None:
                    progress_bar.update(1)
        except KeyError as e:
            print(f"KeyError for {base_filename}: {e}")

In [6]:
# --- SETTINGS ---
main_input_folder = "/media/arnout/Elements/antibioticScreen/"
main_output_folder = "/media/arnout/Elements/antibioticscreenTiff/"
# List all your folders here
selected_raw_folders = ["PLATE1_T7_raw"] 
max_workers = 4 

# --- EXECUTION: One plate at a time ---
for folder in selected_raw_folders:
    input_directory = os.path.join(main_input_folder, folder)
    # Output name: remove "_raw"
    base_output_directory = os.path.join(main_output_folder, folder.replace("_raw", ""))
    
    print(f"\n--- Processing Plate: {folder} ---")
    os.makedirs(base_output_directory, exist_ok=True)
    
    # This runs the files within THIS plate in parallel, 
    # but the loop waits for them to finish before starting the next plate.
    process_nd2_files_parallel(input_directory, base_output_directory, max_workers)
    
    # Clear memory between plates
    gc.collect()

print("\nProcessing completed successfully.")


--- Processing Plate: PLATE1_T7_raw ---


Processing Sample_A6.nd2:   0%|          | 0/100 [00:00<?, ?it/s]


Processing Sample_A6.nd2:   5%|▌         | 5/100 [00:02<00:35,  2.66it/s]



Processing Sample_A6.nd2:   9%|▉         | 9/100 [00:09<03:03,  2.01s/it]



Processing Sample_A6.nd2:   9%|▉         | 9/100 [00:09<03:03,  2.01s/it]



Processing Sample_A6.nd2:  10%|█         | 10/100 [00:12<02:59,  2.00s/it]








Processing Sample_A6.nd2:  14%|█▍        | 14/100 [00:19<03:44,  2.61s/it]








Processing Sample_A6.nd2:  18%|█▊        | 18/100 [00:23<01:45,  1.29s/it]


Processing Sample_A6.nd2:  18%|█▊        | 18/100 [00:23<01:45,  1.29s/it]







Processing Sample_A6.nd2:  20%|██        | 20/100 [00:26<01:54,  1.43s/it]





Processing Sample_A6.nd2:  24%|██▍       | 24/100 [00:29<01:34,  1.24s/it]

Processing Sample_A6.nd2:  24%|██▍       | 24/100 [00:30<01:34,  1.24s/it]



Processing Sample_A6.nd2:  25%|██▌       | 25/100 [00:31<01:38,  1.31s/it]




Processing Sample_A6.nd2:  26%|██▌       | 26/100 [00:39<03:04,


Processing completed successfully.


In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor

# Settings
main_input_folder = "/media/arnout/Elements/antibioticScreen/"
main_output_folder =  "/media/arnout/Elements/antibioticscreenTiff/"
selected_raw_folders = ["PLATE1_T6_raw"]  # Folders to process
max_workers = 4  # Number of parallel workers

# Prepare input/output directories
input_directories = [
    os.path.join(main_input_folder, folder) for folder in selected_raw_folders
]
base_output_directories = [
    os.path.join(main_output_folder, folder[:-4])  # Remove "_raw"
    for folder in selected_raw_folders
]

# Create output directories if missing
for output_dir in base_output_directories:
    os.makedirs(output_dir, exist_ok=True)

# Process each input-output directory pair in parallel
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    for input_directory, base_output_directory in zip(input_directories, base_output_directories):
        executor.submit(process_nd2_files_parallel, input_directory, base_output_directory, max_workers)

print("Processing completed.")

Processing Sample_A10.nd2:   0%|          | 0/100 [00:00<?, ?it/s]



































Processing Sample_A10.nd2:   4%|▍         | 4/100 [00:18<08:52,  5.54s/it]





































































































Processing Sample_A10.nd2:  15%|█▌        | 15/100 [00:59<06:26,  4.54s/it]


































































































































Processing Sample_A10.nd2:  15%|█▌        | 15/100 [01:02<06:26,  4.54s/it]




























Processing Sample_A10.nd2:  15%|█▌        | 15/100 [01:02<06:26,  4.54s/it]






Processing Sample_A10.nd2:  15%|█▌        | 15/100 [01:03<06:26,  4.54s/it]

















Processing Sample_A10.nd2:  15%|█▌        | 15/100 [01:03<06:26,  4.54s/it]


































Processing Sample_A10.nd2:  15%|█▌        | 15/100 [01:03<06:26,  4.54s/it]






























Processing Sample_A10

In [ ]:
import os

# Define the path to the target directory
base_path = r"E:\Bart E-drive\Stationary phase processing"

# Iterate through all items in the directory
for folder_name in os.listdir(base_path):
    folder_path = os.path.join(base_path, folder_name)

    # Check if the item is a directory
    if os.path.isdir(folder_path):
        new_name = folder_name + "_raw"  # Append "_raw"
        new_path = os.path.join(base_path, new_name)

        # Rename the folder
        os.rename(folder_path, new_path)
        print(f'Renamed: "{folder_name}" -> "{new_name}"')

print("Folder renaming completed.")


Renamed: "PLATE43A" -> "PLATE43A_raw"
Renamed: "PLATE43B" -> "PLATE43B_raw"
Renamed: "PLATE45A" -> "PLATE45A_raw"
Renamed: "PLATE45B" -> "PLATE45B_raw"
Renamed: "PLATE47A" -> "PLATE47A_raw"
Renamed: "PLATE47B" -> "PLATE47B_raw"
Renamed: "PLATE49A" -> "PLATE49A_raw"
Renamed: "PLATE49B" -> "PLATE49B_raw"
Renamed: "PLATE51A" -> "PLATE51A_raw"
Renamed: "PLATE51B" -> "PLATE51B_raw"
Renamed: "PLATE53A" -> "PLATE53A_raw"
Renamed: "PLATE53B" -> "PLATE53B_raw"
Renamed: "PLATE55A" -> "PLATE55A_raw"
Renamed: "PLATE55B" -> "PLATE55B_raw"
Renamed: "PLATE57A" -> "PLATE57A_raw"
Renamed: "PLATE57B" -> "PLATE57B_raw"
Renamed: "PLATE59A" -> "PLATE59A_raw"
Renamed: "PLATE59B" -> "PLATE59B_raw"
Renamed: "PLATE61A" -> "PLATE61A_raw"
Renamed: "PLATE61B" -> "PLATE61B_raw"
Renamed: "PLATE63A" -> "PLATE63A_raw"
Renamed: "PLATE63B" -> "PLATE63B_raw"
Renamed: "PLATE65A" -> "PLATE65A_raw"
Renamed: "PLATE65B" -> "PLATE65B_raw"
Renamed: "PLATE67A" -> "PLATE67A_raw"
Renamed: "PLATE67B" -> "PLATE67B_raw"
Renamed: "PL

In [ ]:
import pandas as pd
import glob
import os

# Define the directory path (use raw string r"" to handle backslashes in Windows)
base_path = r"K:\Export Bart\Stationary_phase_screen"

# Find all Excel files in subdirectories
excel_files = glob.glob(os.path.join(base_path, "**", "*.xlsx"), recursive=True)

# List to store individual dataframes
df_list = []

# Loop through files and read them
for file in excel_files:
    try:
        df = pd.read_excel(file, engine="openpyxl")  # Use openpyxl for .xlsx files
        df["Source_File"] = os.path.basename(file)  # Add filename column for tracking
        df_list.append(df)
    except Exception as e:
        print(f"Error reading {file}: {e}")

# Concatenate all dataframes
if df_list:
    final_df = pd.concat(df_list, ignore_index=True)

    # Save to a new Excel file
    output_path = r"K:\Export Bart\Stationary_phase_screen\drive2_combined_data.xlsx"
    final_df.to_excel(output_path, index=False, engine="openpyxl")

    print(f"✅ Combined file saved to: {output_path}")
else:
    print("❌ No Excel files found or failed to read them.")

✅ Combined file saved to: K:\Export Bart\Stationary_phase_screen\drive2_combined_data.xlsx


In [ ]:
import pandas as pd
import glob
import os

# Define the directory path (use raw string r"" to handle backslashes in Windows)
base_path = r"Z:\SET-SG-B013\Shared-SG-008\Bart Steemans\Analysis scripts\keio collection analysis"

# Find all Excel files in subdirectories
excel_files = glob.glob(os.path.join(base_path, "*combined_data.xlsx"), recursive=True)

# List to store individual dataframes
df_list = []

# Loop through files and read them
for file in excel_files:
    try:
        df = pd.read_excel(file, engine="openpyxl")  # Use openpyxl for .xlsx files
        df["Source_File"] = os.path.basename(file)  # Add filename column for tracking
        df_list.append(df)
    except Exception as e:
        print(f"Error reading {file}: {e}")

# Concatenate all dataframes
if df_list:
    final_df = pd.concat(df_list, ignore_index=True)

    # Save to a new Excel file
    output_path = base_path + r"\Metadata_all_plates.xlsx"
    final_df.to_excel(output_path, index=False, engine="openpyxl")

    print(f"✅ Combined file saved to: {output_path}")
else:
    print("❌ No Excel files found or failed to read them.")

✅ Combined file saved to: Z:\SET-SG-B013\Shared-SG-008\Bart Steemans\Analysis scripts\keio collection analysis\Metadata_all_plates.xlsx


In [ ]:
import re
import pandas as pd

# Load the Excel file
file_path = r"Z:/SET-SG-B013/Shared-SG-008/Bart Steemans/Analysis scripts/keio collection analysis/Metadata_all_plates.xlsx"
df = pd.read_excel(file_path, engine="openpyxl")

def sort_plate(val):
    """
    Returns a tuple for sorting the Metadata_plate column.
    PLATE strings (e.g., "PLATE11A" or "PLATE91-3-5B") are processed into:
        (0, (numeric parts as tuple), letter)
    Non-PLATE values (e.g., "AA", "AB", "B") return:
        (1, val)
    """
    if isinstance(val, str) and val.startswith("PLATE"):
        # Extract numeric parts and letter (if available)
        m = re.match(r'PLATE((?:\d+(?:-\d+)*)?)([AB]?)$', val)
        if m:
            num_str = m.group(1)
            letter = m.group(2)
            if num_str:
                # Create a tuple of integers from the numeric parts.
                num_tuple = tuple(int(x) for x in num_str.split("-"))
            else:
                # If no number is present, place these after numbered plates.
                num_tuple = (float('inf'),)
            return (0, num_tuple, letter)
    # For non-PLATE values, sort alphabetically.
    return (1, val)

def sort_well(well):
    """
    Returns a tuple for sorting the Metadata_well column.
    It expects well values like "A1", "B8", etc. The function extracts the
    letter part and converts the numeric part into an integer.
    """
    m = re.match(r'^([A-F])(\d+)$', str(well).strip())
    if m:
        letter = m.group(1)
        num = int(m.group(2))
        return (letter, num)
    # In case of unexpected format, fallback to the raw value.
    return (well, 0)

# Create helper key columns for sorting
df['plate_key'] = df['Metadata_plate'].apply(sort_plate)
df['well_key'] = df['Metadata_well'].apply(sort_well)

# Sort the DataFrame by plate first, then by well.
df_sorted = df.sort_values(by=['plate_key', 'well_key']).drop(columns=['plate_key', 'well_key'])

# To check the sorted DataFrame:
print(df_sorted.head())
df_sorted.to_excel(r"Z:/SET-SG-B013/Shared-SG-008/Bart Steemans/Analysis scripts/keio collection analysis/Metadata_all_plates_sorted.xlsx", index=False, engine="openpyxl")

   Metadata_plate Metadata_well Metadata_gene Metadata_Comments  \
48        PLATE1A            A1          hchA               NaN   
49        PLATE1A            A2             0               NaN   
50        PLATE1A            A3           ada               NaN   
51        PLATE1A            A4          adiY               NaN   
52        PLATE1A            A5          appY               NaN   

                  Source_File  
48  drive1_combined_data.xlsx  
49  drive1_combined_data.xlsx  
50  drive1_combined_data.xlsx  
51  drive1_combined_data.xlsx  
52  drive1_combined_data.xlsx  


: 

In [ ]:
import os

# Define paths
source_path = r"E:\Bart E-drive\Stationary phase processing"
destination_path = r"F:\Export Bart\Stationary_phase_screen"

# Ensure destination path exists
os.makedirs(destination_path, exist_ok=True)

# Iterate through the source directory
for folder_name in os.listdir(source_path):
    folder_path = os.path.join(source_path, folder_name)

    # Process only directories ending in "_raw"
    if os.path.isdir(folder_path) and folder_name.endswith("_raw"):
        new_folder_name = folder_name[:-4]  # Remove "_raw"
        new_folder_path = os.path.join(destination_path, new_folder_name)

        # Create the folder in the destination path
        os.makedirs(new_folder_path, exist_ok=True)
        print(f'Created: "{new_folder_name}" in "{destination_path}"')

print("Folder creation completed.")


Created: "PLATE43A" in "F:\Export Bart\Stationary_phase_screen"
Created: "PLATE43B" in "F:\Export Bart\Stationary_phase_screen"
Created: "PLATE45A" in "F:\Export Bart\Stationary_phase_screen"
Created: "PLATE45B" in "F:\Export Bart\Stationary_phase_screen"
Created: "PLATE47A" in "F:\Export Bart\Stationary_phase_screen"
Created: "PLATE47B" in "F:\Export Bart\Stationary_phase_screen"
Created: "PLATE49A" in "F:\Export Bart\Stationary_phase_screen"
Created: "PLATE49B" in "F:\Export Bart\Stationary_phase_screen"
Created: "PLATE51A" in "F:\Export Bart\Stationary_phase_screen"
Created: "PLATE51B" in "F:\Export Bart\Stationary_phase_screen"
Created: "PLATE53A" in "F:\Export Bart\Stationary_phase_screen"
Created: "PLATE53B" in "F:\Export Bart\Stationary_phase_screen"
Created: "PLATE55A" in "F:\Export Bart\Stationary_phase_screen"
Created: "PLATE55B" in "F:\Export Bart\Stationary_phase_screen"
Created: "PLATE57A" in "F:\Export Bart\Stationary_phase_screen"
Created: "PLATE57B" in "F:\Export Bart\S

In [ ]:
import os
import shutil

# Define the parent directory
parent_dir = "F:/Brightfield and phase training data/matt data/06 03 2025/250306_Ecoli"

# Get all subfolders in the directory
subfolders = [os.path.join(parent_dir, f) for f in os.listdir(parent_dir) if os.path.isdir(os.path.join(parent_dir, f))]

# Move files from subfolders to the parent directory
for subfolder in subfolders:
    for file_name in os.listdir(subfolder):
        file_path = os.path.join(subfolder, file_name)
        new_path = os.path.join(parent_dir, file_name)

        # Ensure no overwriting issues
        if os.path.exists(new_path):
            base, ext = os.path.splitext(file_name)
            counter = 1
            while os.path.exists(new_path):
                new_path = os.path.join(parent_dir, f"{base}_{counter}{ext}")
                counter += 1
        
        # Move the file
        shutil.move(file_path, new_path)

print("Files moved safely! No folders were deleted.")


Files moved safely! No folders were deleted.


## Change filenames

In [ ]:
import os

# Directory path containing the images
directory = 'F:/brightfield_model/train/'

# Iterate through all files in the directory
for filename in os.listdir(directory):
    if filename.endswith(".tif") and "C1_cp_masks" in filename:
        # Construct the new filename
        new_filename = filename.replace("C1_cp_masks", "C2_masks")
        
        # Join the directory path with the filename
        old_filepath = os.path.join(directory, filename)
        new_filepath = os.path.join(directory, new_filename)
        
        # Rename the file
        os.rename(old_filepath, new_filepath)
        print(f"Renamed {filename} to {new_filename}")

## Create patches from large images

In [ ]:
import os
import glob
import numpy as np
import tifffile as tiff

def split_image(image_path, output_directory, patch_size, suffix):
    # Read the image
    img = tiff.imread(image_path)
    
    # Extract the filename without extension
    filename = os.path.splitext(os.path.basename(image_path))[0]
    
    # Calculate the number of patches
    img_height, img_width = img.shape[:2]
    num_patches_y = img_height // patch_size
    num_patches_x = img_width // patch_size
    
    patch_num = 0
    
    # Create patches
    for y in range(num_patches_y):
        for x in range(num_patches_x):
            patch = img[y*patch_size:(y+1)*patch_size, x*patch_size:(x+1)*patch_size]
            patch_filename = f"{filename}_{patch_num}{suffix}.tif"
            patch_filename = patch_filename.replace('masks_', '', 1)
            patch_path = os.path.join(output_directory, patch_filename)
            tiff.imwrite(patch_path, patch)
            patch_num += 1

def process_images(input_directory, output_directory, patch_size):
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)
    
    image_paths = glob.glob(os.path.join(input_directory, "*_C2.tiff"))
    mask_paths = glob.glob(os.path.join(input_directory, "*_C2_masks.tif"))
    
    for image_path in image_paths:
        split_image(image_path, output_directory, patch_size, '_C2')
        
    for mask_path in mask_paths:
        split_image(mask_path, output_directory, patch_size, '_C2_masks')

# Define the input and output directories and patch size
input_directory = "F:/brightfield_model/traino"
output_directory = "F:/brightfield_model/patcheso"
patch_size = 512  # Define your patch size here

# Process the images
process_images(input_directory, output_directory, patch_size)


## Cleaning patches with wrong dimensions and not enough masks

In [ ]:
import os
from PIL import Image
import numpy as np

# Folder containing your patches
folder = 'F:/brightfield_model/patches'

# Define the desired shape
desired_shape = (512, 512)

# Get all files in the directory
all_files = os.listdir(folder)
images = [f for f in all_files if f.endswith('.tif') and not f.endswith('_masks.tif')]
masks = [f for f in all_files if f.endswith('_masks.tif')]

# Function to check if image/mask has the correct shape
def has_correct_shape(file_path):
    with Image.open(file_path) as img:
        return img.size == desired_shape

# Function to count the number of labels (masks) in the mask image
def count_labels(mask_file_path):
    with Image.open(mask_file_path) as img:
        mask_array = np.array(img)
        return len(np.unique(mask_array)) - 1  # -1 to exclude background if present

# Function to remove a file
def remove_file(file_path):
    os.remove(file_path)
    print(f"Removed: {file_path}")

# Tracking variables
cleaned_images = 0
cleaned_masks = 0
reasons = []

for img_file in images:
    base_name = img_file.split('.')[0]
    corresponding_masks = [m for m in masks if m.startswith(base_name)]

    img_path = os.path.join(folder, img_file)
    
    # Check if the image has the correct shape
    if not has_correct_shape(img_path):
        remove_file(img_path)
        cleaned_images += 1
        reasons.append(f"Image '{img_file}' removed because its shape is incorrect.")
        
        # Remove corresponding masks
        for mask in corresponding_masks:
            mask_path = os.path.join(folder, mask)
            remove_file(mask_path)
            cleaned_masks += 1
        continue

    # Check the number of labels in the masks
    if corresponding_masks:
        num_labels = count_labels(os.path.join(folder, corresponding_masks[0]))
        if num_labels < 5:
            remove_file(img_path)
            cleaned_images += 1
            reasons.append(f"Image '{img_file}' removed because it has fewer than 15 masks.")
            
            # Remove corresponding masks
            for mask in corresponding_masks:
                mask_path = os.path.join(folder, mask)
                remove_file(mask_path)
                cleaned_masks += 1
        else:
            all_masks_correct = True
            for mask in corresponding_masks:
                mask_path = os.path.join(folder, mask)
                if not has_correct_shape(mask_path):
                    all_masks_correct = False
                    break
            
            if not all_masks_correct:
                remove_file(img_path)
                cleaned_images += 1
                reasons.append(f"Image '{img_file}' removed because at least one corresponding mask has incorrect shape.")
                
                # Remove corresponding masks
                for mask in corresponding_masks:
                    mask_path = os.path.join(folder, mask)
                    remove_file(mask_path)
                    cleaned_masks += 1

# Print summary
print(f"Total images removed: {cleaned_images}")
print(f"Total masks removed: {cleaned_masks}")
print("Reasons for removal:")
for reason in reasons:
    print(reason)


Removed: F:/brightfield_model/patches\R2_Ab_XY0_C2_1_C2.tif
Removed: F:/brightfield_model/patches\R2_Ab_XY0_C2_1_C2_masks.tif
Removed: F:/brightfield_model/patches\R2_Ab_XY0_C2_22_C2.tif
Removed: F:/brightfield_model/patches\R2_Ab_XY0_C2_22_C2_masks.tif
Removed: F:/brightfield_model/patches\R2_Ab_XY0_C2_23_C2.tif
Removed: F:/brightfield_model/patches\R2_Ab_XY0_C2_23_C2_masks.tif
Removed: F:/brightfield_model/patches\R2_Ab_XY0_C2_2_C2.tif
Removed: F:/brightfield_model/patches\R2_Ab_XY0_C2_2_C2_masks.tif
Removed: F:/brightfield_model/patches\R2_Ab_XY1_C2_15_C2.tif
Removed: F:/brightfield_model/patches\R2_Ab_XY1_C2_15_C2_masks.tif
Removed: F:/brightfield_model/patches\R2_Ab_XY1_C2_4_C2.tif
Removed: F:/brightfield_model/patches\R2_Ab_XY1_C2_4_C2_masks.tif
Removed: F:/brightfield_model/patches\R2_Ab_XY2_C2_11_C2.tif
Removed: F:/brightfield_model/patches\R2_Ab_XY2_C2_11_C2_masks.tif
Removed: F:/brightfield_model/patches\R2_Ab_XY2_C2_1_C2.tif
Removed: F:/brightfield_model/patches\R2_Ab_XY2_C2

In [ ]:
import os
import numpy as np
from skimage import io
from tifffile import imread
from sklearn.metrics import jaccard_score
from tqdm import tqdm

# Paths to the directories containing the images
gt_path = 'F:/brightfield_model/train/'
pred_path = 'F:/brightfield_model/brightfield/masks/'

# Function to calculate the Jaccard index between two images
def calculate_jaccard_index(image1, image2):
    # Flatten the images to 1D arrays for comparison
    image1_flat = image1.flatten()
    image2_flat = image2.flatten()
    # Calculate the Jaccard index using sklearn's jaccard_score
    return jaccard_score(image1_flat, image2_flat, average='binary')

# Get list of all files in the ground truth and prediction directories
gt_files = [f for f in os.listdir(gt_path) if f.endswith('_masks.tif')]
pred_files = [f for f in os.listdir(pred_path) if f.endswith('_cp_masks.tif')]

# Initialize a list to store Jaccard indices for each pair
jaccard_indices = []

# Calculate the Jaccard index for each corresponding pair of images
for gt_file in tqdm(gt_files, desc="Calculating Jaccard Index"):
    # Construct the corresponding predicted image file name
    pred_file = gt_file.replace('_masks.tif', '_cp_masks.tif')
    
    # Check if the corresponding predicted image exists
    if pred_file in pred_files:
        # Load the ground truth and predicted images
        gt_image = imread(os.path.join(gt_path, gt_file))
        pred_image = imread(os.path.join(pred_path, pred_file))
        
        # Ensure both images are binary (i.e., contain only 0 and 1)
        gt_image = gt_image > 0
        pred_image = pred_image > 0
        
        # Calculate the Jaccard index
        jaccard_index = calculate_jaccard_index(gt_image, pred_image)
        
        # Store the result
        jaccard_indices.append(jaccard_index)

# Display the Jaccard index for each pair
for file_name, jaccard_index in zip(gt_files, jaccard_indices):
    print(f"Jaccard Index for {file_name}: {jaccard_index:.4f}")

# Calculate and display the average Jaccard index
if jaccard_indices:
    average_jaccard = np.mean(jaccard_indices)
    print(f"\nAverage Jaccard Index: {average_jaccard:.4f}")


Calculating Jaccard Index: 100%|██████████| 36/36 [01:12<00:00,  2.01s/it]

Jaccard Index for R2_Ab_XY0_C2_masks.tif: 0.8984
Jaccard Index for R2_Ab_XY1_C2_masks.tif: 0.9100
Jaccard Index for R2_Ab_XY2_C2_masks.tif: 0.9145
Jaccard Index for R2_Ab_XY3_C2_masks.tif: 0.8571
Jaccard Index for R2_Ab_XY4_C2_masks.tif: 0.9062
Jaccard Index for R2_Ab_XY5_C2_masks.tif: 0.8835
Jaccard Index for R2_Ab_XY6_C2_masks.tif: 0.8665
Jaccard Index for R2_Ab_XY7_C2_masks.tif: 0.9074
Jaccard Index for R2_Ab_XY8_C2_masks.tif: 0.9053
Jaccard Index for R2_Ds_XY0_C2_masks.tif: 0.9166
Jaccard Index for R2_Ds_XY1_C2_masks.tif: 0.8942
Jaccard Index for R2_Ds_XY2_C2_masks.tif: 0.8655
Jaccard Index for R2_Ds_XY3_C2_masks.tif: 0.9142
Jaccard Index for R2_Ds_XY4_C2_masks.tif: 0.9124
Jaccard Index for R2_Ds_XY5_C2_masks.tif: 0.9136
Jaccard Index for R2_Ds_XY6_C2_masks.tif: 0.9104
Jaccard Index for R2_Ds_XY7_C2_masks.tif: 0.9149
Jaccard Index for R2_Ds_XY8_C2_masks.tif: 0.8510
Jaccard Index for R2_Ef05_XY0_C2_masks.tif: 0.9309
Jaccard Index for R2_Ef05_XY1_C2_masks.tif: 0.8583
Jaccard Index fo

# Code for single file nd2 files

In [ ]:
import os
import gc
import concurrent.futures
from tqdm import tqdm
from nd2reader import ND2Reader
from PIL import Image

def export_channel(nd2_filepath, output_directory, channel_idx, width, height, progress_bar=None):
    with ND2Reader(nd2_filepath) as images:
        images.default_coords['c'] = channel_idx  # Set the channel
        try:
            frame = images.get_frame_2D(c=channel_idx)  # No field index needed
            if frame.shape == (height, width):
                base_filename = os.path.basename(nd2_filepath)
                filename_without_ext = os.path.splitext(base_filename)[0]
                output_filename = f"{filename_without_ext}_C{channel_idx + 1}.tiff"
                output_filepath = os.path.join(output_directory, output_filename)
                Image.fromarray(frame).save(output_filepath)
                if progress_bar is not None:
                    progress_bar.update(1)
            else:
                print(f"Skipping invalid frame for channel {channel_idx}")
        except KeyError as e:
            print(f"KeyError for channel {channel_idx}: {e}")

def export_channels_to_tiff_parallel(nd2_filepath, output_directory, max_workers=2):
    os.makedirs(output_directory, exist_ok=True)
    with ND2Reader(nd2_filepath) as images:
        num_channels = images.sizes.get('c', 1)
        width = images.sizes.get('x', None)
        height = images.sizes.get('y', None)

        with tqdm(total=num_channels, desc=f"Processing {os.path.basename(nd2_filepath)}") as progress_bar:
            with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
                futures = [
                    executor.submit(export_channel, nd2_filepath, output_directory, channel_idx, width, height, progress_bar)
                    for channel_idx in range(num_channels)
                ]
                for future in concurrent.futures.as_completed(futures):
                    future.result()
    
    gc.collect()  # Free memory explicitly